# Stellar Classification: Baseline Modeling Pipeline

This notebook implements a cross-validated baseline modeling pipeline using LightGBM, XGBoost, and CatBoost. Our primary objective is to evaluate model validation performance directly on the Balanced Accuracy metric while testing feature transformations engineered to address spatial coordinate wrap-around and magnitude scaling.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

## 1. Configurations, Constants & Environment Routing

We configure our core execution variables. Setting $K = 5$ for cross-validation splits strikes a robust bias-variance trade-off on $577\text{k}$ samples. 
Target codes and categoricals are mapped to integer sequences to align with the array representations required by gradient boosted decision trees.

In [2]:
# Dynamic path resolution for Local vs Kaggle environment
if os.path.exists('/kaggle/input/competitions/playground-series-s6e6'):
    DATA_DIR = '/kaggle/input/competitions/playground-series-s6e6'
    PRED_DIR = './predictions'
else:
    DATA_DIR = '../data'
    PRED_DIR = '../predictions'

N_SPLITS = 5
SEED = 42

# Target mapping variables
TARGET_MAP = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
INV_TARGET_MAP = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}

# Categorical mapping variables
SPECTRAL_MAP = {'M': 0, 'A/F': 1, 'G/K': 2, 'O/B': 3}
GALAXY_POP_MAP = {'Red_Sequence': 0, 'Blue_Cloud': 1}

print(f"Data Directory: {DATA_DIR}")
print(f"Predictions Directory: {PRED_DIR}")

Data Directory: /kaggle/input/competitions/playground-series-s6e6
Predictions Directory: ./predictions


## 2. Mathematically Structured Feature Engineering

To prepare raw data for the modeling pipeline, we implement three key transformations:
1. **10 Astronomical Color Indices:** Subtraction combinations of all magnitude bands ($u, g, r, i, z$). This removes distance-dependent scaling variations and serves as distance-invariant estimators of radiation temperatures.
2. **3D Cartesian Projection:** Transforms angular coordinates (\alpha, \delta) into spatial coordinates on a 3D unit sphere, resolving the $0^\circ / 360^\circ$ right ascension wrapping discontinuity.
3. **Ordinal Categorical Mapping:** Encoding `spectral_type` and `galaxy_population` categories into numerical levels to allow smooth tree splits.

In [3]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Applies feature engineering to astronomical features.

    Args:
        df: Input raw DataFrame.

    Returns:
        DataFrame with mapped categoricals, colors, and 3D coordinates.
    """
    df = df.copy()
    
    # Categoricals mapping
    df['spectral_type_code'] = df['spectral_type'].map(SPECTRAL_MAP)
    df['galaxy_pop_code'] = df['galaxy_population'].map(GALAXY_POP_MAP)
    
    # Astronomical Colors (differences in magnitudes)
    # Successive filters
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    
    # Non-successive filters
    df['u_r'] = df['u'] - df['r']
    df['g_i'] = df['g'] - df['i']
    df['r_z'] = df['r'] - df['z']
    df['u_i'] = df['u'] - df['i']
    df['g_z'] = df['g'] - df['z']
    df['u_z'] = df['u'] - df['z']
    
    # Trigonometric transformation of celestial coordinates (Right Ascension & Declination)
    alpha_rad = np.radians(df['alpha'])
    delta_rad = np.radians(df['delta'])
    df['coord_x'] = np.cos(delta_rad) * np.cos(alpha_rad)
    df['coord_y'] = np.cos(delta_rad) * np.sin(alpha_rad)
    df['coord_z'] = np.sin(delta_rad)
    
    # Drop raw IDs and raw categorical columns
    cols_to_drop = ['id', 'spectral_type', 'galaxy_population']
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
    
    return df

## 3. Stratified Cross-Validation & Metric-Weight Alignment

To evaluate baseline models without data leakage, we structure a 5-fold Stratified Cross-Validation scheme. This guarantees that each fold accurately mirrors the overall target class imbalance.

### Class Weight Adjustment
Because the objective is **Balanced Accuracy**, standard classifiers need adjustments to avoid biased predictions on the majority class. We calibrate weights inversely proportional to target frequencies:

$$\text{Weight}_c = \frac{\text{Total Samples}}{C \times \text{Samples}_c}$$

Each gradient boosting library implements sample/class weights differently:
- **LightGBM:** Configured via `class_weight='balanced'` which internally applies inverse target frequency weights.
- **CatBoost:** Managed using `auto_class_weights='Balanced'` to balance gradient step updates.
- **XGBoost:** Lacks automatic class balancing for multi-class classifiers. We compute custom sample weights per fold using `sklearn.utils.class_weight.compute_sample_weight('balanced', y_train)` and pass them to the `fit` method.

In [4]:
def get_model(model_name: str, seed: int = 42):
    """Initializes the classifier model.
    """
    if model_name == 'lgb':
        return lgb.LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.05,
            num_leaves=63,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight='balanced',
            random_state=seed,
            n_jobs=-1,
            verbose=-1
        )
    elif model_name == 'xgb':
        return xgb.XGBClassifier(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=7,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method='hist',
            random_state=seed,
            n_jobs=-1
        )
    elif model_name == 'cat':
        return CatBoostClassifier(
            iterations=1000,
            learning_rate=0.05,
            depth=7,
            auto_class_weights='Balanced',
            random_state=seed,
            thread_count=-1,
            verbose=False
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")

def train_cv(model_name: str, n_splits: int = 5, seed: int = 42) -> tuple[np.ndarray, np.ndarray, float]:
    """Trains a model using Stratified K-Fold CV.
    """
    print(f"\n--- Training {model_name.upper()} model using {n_splits}-fold Stratified CV ---")
    
    train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
    test_raw = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
    
    # Preprocess features
    X = feature_engineering(train_raw)
    y = X.pop('class').map(TARGET_MAP)
    X_test = feature_engineering(test_raw)
    
    print(f"Feature columns: {list(X.columns)}")
    print(f"Training shape: {X.shape}, Test shape: {X_test.shape}")
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    oof_preds = np.zeros((len(X), 3))
    test_preds = np.zeros((len(X_test), 3))
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        model = get_model(model_name, seed)
        
        if model_name == 'lgb':
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
            )
        elif model_name == 'xgb':
            sample_weight = compute_sample_weight('balanced', y_train)
            model.fit(
                X_train, y_train,
                sample_weight=sample_weight,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
        elif model_name == 'cat':
            model.fit(
                X_train, y_train,
                eval_set=(X_val, y_val),
                early_stopping_rounds=50,
                verbose=False
            )
            
        # Predict validation probabilities
        val_preds_prob = model.predict_proba(X_val)
        oof_preds[val_idx] = val_preds_prob
        
        # Score fold
        val_preds = np.argmax(val_preds_prob, axis=1)
        score = balanced_accuracy_score(y_val, val_preds)
        fold_scores.append(score)
        print(f"Fold {fold+1} Balanced Accuracy: {score:.5f}")
        
        # Test predictions accumulation
        test_preds += model.predict_proba(X_test) / n_splits
        
    cv_score = balanced_accuracy_score(y, np.argmax(oof_preds, axis=1))
    print(f"\n{model_name.upper()} CV Balanced Accuracy Score: {cv_score:.5f}")
    
    # Save out prediction probabilities for blending
    os.makedirs(PRED_DIR, exist_ok=True)
    np.save(os.path.join(PRED_DIR, f'{model_name}_oof.npy'), oof_preds)
    np.save(os.path.join(PRED_DIR, f'{model_name}_test.npy'), test_preds)
    
    return oof_preds, test_preds, cv_score

## 4. Diverse Gradient Boosting Runs

We train three distinct gradient boosting architectures to ensure structural diversity, which helps when ensembling OOF predictions later:
1. **LightGBM:** Leaf-wise (best-first) growth splits nodes with maximum loss reduction, creating deeper, asymmetric trees.
2. **XGBoost:** Level-wise (depth-first) growth splits all nodes at a given depth concurrently, regularized via post-pruning parameters.
3. **CatBoost:** Builds oblivious symmetric trees using the same splitting criterion across an entire tree level, reducing variance and mitigating overfitting.

In [5]:
lgb_oof, lgb_test, lgb_score = train_cv('lgb', n_splits=N_SPLITS, seed=SEED)


--- Training LGB model using 5-fold Stratified CV ---
Feature columns: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type_code', 'galaxy_pop_code', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'r_z', 'u_i', 'g_z', 'u_z', 'coord_x', 'coord_y', 'coord_z']
Training shape: (577347, 23), Test shape: (247435, 23)
Fold 1 Balanced Accuracy: 0.96549
Fold 2 Balanced Accuracy: 0.96557
Fold 3 Balanced Accuracy: 0.96474
Fold 4 Balanced Accuracy: 0.96468
Fold 5 Balanced Accuracy: 0.96511

LGB CV Balanced Accuracy Score: 0.96512


In [6]:
xgb_oof, xgb_test, xgb_score = train_cv('xgb', n_splits=N_SPLITS, seed=SEED)


--- Training XGB model using 5-fold Stratified CV ---
Feature columns: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type_code', 'galaxy_pop_code', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'r_z', 'u_i', 'g_z', 'u_z', 'coord_x', 'coord_y', 'coord_z']
Training shape: (577347, 23), Test shape: (247435, 23)
Fold 1 Balanced Accuracy: 0.96599
Fold 2 Balanced Accuracy: 0.96582
Fold 3 Balanced Accuracy: 0.96453
Fold 4 Balanced Accuracy: 0.96510
Fold 5 Balanced Accuracy: 0.96537

XGB CV Balanced Accuracy Score: 0.96536


In [7]:
cat_oof, cat_test, cat_score = train_cv('cat', n_splits=N_SPLITS, seed=SEED)


--- Training CAT model using 5-fold Stratified CV ---
Feature columns: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type_code', 'galaxy_pop_code', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'r_z', 'u_i', 'g_z', 'u_z', 'coord_x', 'coord_y', 'coord_z']
Training shape: (577347, 23), Test shape: (247435, 23)
Fold 1 Balanced Accuracy: 0.96218
Fold 2 Balanced Accuracy: 0.96255
Fold 3 Balanced Accuracy: 0.96148
Fold 4 Balanced Accuracy: 0.96197
Fold 5 Balanced Accuracy: 0.96236

CAT CV Balanced Accuracy Score: 0.96211


## 5. Summary of Baseline Results

Compare validation scores across models. Out-of-fold probability files are written to `/predictions` for ensembling.

In [8]:
print("=== Baseline CV Scores (Balanced Accuracy) ===")
print(f"LightGBM: {lgb_score:.5f}")
print(f"XGBoost:  {xgb_score:.5f}")
print(f"CatBoost: {cat_score:.5f}")

=== Baseline CV Scores (Balanced Accuracy) ===
LightGBM: 0.96512
XGBoost:  0.96536
CatBoost: 0.96211
